In [1]:
# ! python -m ipykernel install --user --name FoodDeliveryPipeline --display-name "FoodDeliveryPipeline (.venv)"

In [2]:
spark.stop()

NameError: name 'spark' is not defined

In [1]:
import warnings
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import *
from pyspark.sql.types import *
import pyspark.sql.dataframe
from datetime import datetime
from IPython.display import display
import pandas as pd
import sys
import os

sys.path.append(os.path.abspath("../.."))
from utils.kafka_config import KafkaConfig
import yaml
from pathlib import Path
# Configure pandas to show ful output without truncation
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_rows', None)     # Show all rows
pd.set_option('display.max_colwidth', None) # Don't truncate columns content
pd.set_option('display.width', None)        # Use full width

print(" Libraries imported successfully")
warnings.filterwarnings("ignore")


 Libraries imported successfully


In [ ]:
CONFIG = KafkaConfig()._get_config_json()

TOPIC_TO_TABLES = CONFIG.get("topic_to_table")
KAFAK_BOOTSTRAP_SERVER = CONFIG.get("kafka").get("bootstrap_servers")
SUBSCRIBE_PATTERN = CONFIG.get("kafka").get("subscribe_pattern")
TRIGGER_INTERVAL = CONFIG.get("streaming").get("trigger_interval")
CHECKPOINT_LOCATION = CONFIG.get("streaming").get("checkpoint_location")
STARTING_OFFSETS = CONFIG.get("streaming").get("starting_offsets")


In [3]:
try: 
    from pyspark import SparkContext
    sc = SparkContext._active_spark_context
    if sc: 
        sc.stop()
        print(" Stoped previous SparkContext")
except:
    pass
    
spark = (
    SparkSession
    .builder
    .appName("Streaming from Kafka")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config("spark.sql.shuffle.partitions", 4)
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3"
    )
   .master("local[*]")
    .getOrCreate()
)
spark

:: loading settings :: url = jar:file:/usr/local/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/spark/.ivy2.5.2/cache
The jars for the packages stored in: /home/spark/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-946501b0-f8e2-4639-99bd-bb05f0bead1a;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.3 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.3 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.5 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.c

ConnectionRefusedError: [Errno 111] Connection refused

In [3]:
spark.conf.set("spark.sql.debug.maxToStringFields", "1000")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.debug.maxToStringFields", "1000")

spark.conf.set("spark.hadoop.fs.s3a.endpoint", "http://minio-lb:9000")
spark.conf.set("spark.hadoop.fs.s3a.access.key", "minioadmin")
spark.conf.set("spark.hadoop.fs.s3a.secret.key", "minioadmin123")

In [4]:
df_json = (spark
      .read
      .option("multiline", "true")
      .json("./test_ds/*.json")

     )
df_json.show(truncate=True)

+--------------------+------+---+--------------------+-----------+-------------+
|               after|before| op|              source|transaction|        ts_ms|
+--------------------+------+---+--------------------+-----------+-------------+
|{NULL, Will defin...|  NULL|  c|{postgresql, mast...|       NULL|1778935609059|
|{99.5, NULL, 2026...|  NULL|  c|{postgresql, mast...|       NULL|1778935985618|
+--------------------+------+---+--------------------+-----------+-------------+



In [24]:
def print_as_df(df, limit = 5):
    if isinstance(df, pyspark.sql.dataframe.DataFrame):
         display(df.limit(limit).toPandas())
    else:
        print('Unknow types. Just spark dataframe is acceptable')

In [28]:
df_json = (spark
      .read
      .parquet("./test_ds/*.parquet")

     )

print_as_df(df_json)

,operation,source_ts_ms,source_lsn,source_txid,source_table,source_snapshot,before_payload,after_payload,kafka_topic,kafka_partition,kafka_offset,kafka_timestamp,ingestion_ts
0,u,1778935985513,29951188016,5979347,drivers,false,None,"{""driver_id"":1669,""full_name"":""Mishal Al-Harbi"",""phone"":""+966551952999"",""vehicle_type"":""scooter"",""city_id"":3,""status"":""offline"",""onboarded_at"":""2025-07-18T12:58:17.506058Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-16T12:53:05.510185Z""}",cdc.public.drivers,0,0,2026-05-16 12:53:07.327,2026-05-16 14:19:20.970
1,u,1778935985523,29951190056,5979351,drivers,false,None,"{""driver_id"":4818,""full_name"":""Naif Al-Khateeb"",""phone"":""+966552978242"",""vehicle_type"":""scooter"",""city_id"":5,""status"":""available"",""onboarded_at"":""2025-08-14T12:47:19.521240Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-16T12:53:05.522546Z""}",cdc.public.drivers,0,1,2026-05-16 12:53:07.327,2026-05-16 14:19:20.970
2,u,1778935985529,29951193376,5979355,drivers,false,None,"{""driver_id"":10216,""full_name"":""Khalid Al-Mansour"",""phone"":""+966525968572"",""vehicle_type"":""bike"",""city_id"":4,""status"":""offline"",""onboarded_at"":""2024-12-29T07:06:29.547084Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-16T12:53:05.529148Z""}",cdc.public.drivers,0,2,2026-05-16 12:53:07.327,2026-05-16 14:19:20.970
3,u,1778935985533,29951194368,5979357,drivers,false,None,"{""driver_id"":11832,""full_name"":""Saleh Al-Zahrani"",""phone"":""+966528235668"",""vehicle_type"":""scooter"",""city_id"":5,""status"":""offline"",""onboarded_at"":""2025-05-21T10:15:13.554488Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-16T12:53:05.532450Z""}",cdc.public.drivers,0,3,2026-05-16 12:53:07.327,2026-05-16 14:19:20.970
4,u,1778935995594,29953427688,5979906,drivers,false,None,"{""driver_id"":3097,""full_name"":""Dalia Al-Saadi"",""phone"":""+966530571311"",""vehicle_type"":""car"",""city_id"":1,""status"":""available"",""onboarded_at"":""2025-02-16T21:43:41.511895Z"",""is_active"":true,""created_at"":""2026-05-12T20:07:56.477834Z"",""updated_at"":""2026-05-16T12:53:15.594123Z""}",cdc.public.drivers,0,4,2026-05-16 12:53:15.919,2026-05-16 14:19:20.970


In [50]:
kafka_offset_max = df_json.select(max("kafka_offset")).collect()
kafka_offset_max

[Row(max(kafka_offset)=61)]

In [4]:
CONFIG =  KafkaConfig(config_path='../../conf/topic_mapping.yaml')._get_config_json()
TOPIC_TO_TABLES = CONFIG.get("topic_to_table")
KAFAK_BOOTSTRAP_SERVER = CONFIG.get("kafka").get("bootstrap_servers")
SUBSCRIBE_PATTERN = CONFIG.get("kafka").get("subscribe_pattern")
TRIGGER_INTERVAL = CONFIG.get("streaming").get("trigger_interval")
CHECKPOINT_LOCATION = CONFIG.get("streaming").get("checkpoint_location")
STARTING_OFFSETS = CONFIG.get("streaming").get("starting_offsets")

{'topic_to_table': {'cdc.public.orders': 'lake.bronze.orders', 'cdc.public.order_items': 'lake.bronze.order_items', 'cdc.public.payments': 'lake.bronze.payments', 'cdc.public.order_status_events': 'lake.bronze.order_status_events', 'cdc.public.reviews': 'lake.bronze.reviews', 'cdc.public.drivers': 'lake.bronze.drivers', 'cdc.public.menu_items': 'lake.bronze.menu_items'}, 'kafka': {'bootstrap_servers': 'kafka:9092', 'subscribe_pattern': 'cdc\\.public\\..*'}, 'streaming': {'trigger_interval': '30 seconds', 'checkpoint_location': 's3a://lake/checkpoints/bronze_writer', 'starting_offsets': 'earliest'}}


In [16]:
SUBSCRIBE_PATTERN.replace("*","")

'cdc\\.public\\..*'

In [10]:
spark.conf.set("spark.sql.streaming.checkpointLocation", CHECKPOINT_LOCATION)

In [11]:
kafka_df = (
    spark
        .readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFAK_BOOTSTRAP_SERVER)
        .option("subscribePattern", SUBSCRIBE_PATTERN)
        .option("startingOffsets", STARTING_OFFSETS)
        .option("failOnDataLoss", "false")
        .load()
)

In [12]:
kafka_df.printSchema()


root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [13]:
schema_source = schema = StructType([

    StructField("ts_ms", LongType(), True),
    StructField("lsn", LongType(), True),
    StructField("txId", LongType(), True),
    StructField("table", StringType(), True),
    StructField("snapshot", StringType(), True),

])
schema = StructType([

    StructField("op", StringType(), True),
    StructField("before", StringType(), True),
    StructField("after", StringType(), True),
    StructField("source", schema_source, True), 
    StructField("ts_ms", LongType(), True),
])

In [14]:
kafka_json_df = kafka_df.withColumn("value", expr("cast(value as string)"))
kafka_json_df.printSchema()

root
 |-- key: binary (nullable = true)
 |-- value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [15]:
parsed_kafka = kafka_json_df.withColumn("value_json", from_json(col("value"), schema))
parsed_kafka.printSchema()                             

root
 |-- key: binary (nullable = true)
 |-- value: string (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)
 |-- value_json: struct (nullable = true)
 |    |-- op: string (nullable = true)
 |    |-- before: string (nullable = true)
 |    |-- after: string (nullable = true)
 |    |-- source: struct (nullable = true)
 |    |    |-- ts_ms: long (nullable = true)
 |    |    |-- lsn: long (nullable = true)
 |    |    |-- txId: long (nullable = true)
 |    |    |-- table: string (nullable = true)
 |    |    |-- snapshot: string (nullable = true)
 |    |-- ts_ms: long (nullable = true)



In [16]:
flatened_event = parsed_kafka.select(
                            col("topic").alias("kafka_topic"),
                            col("partition").alias("kafka_partition"),
                            col("offset").alias("kafka_offset"),
                            col("timestamp").alias("kafka_timestamp"),                                
                            col("value_json.op").alias("operation"),
                            col("value_json.before").alias("before"),
                            col("value_json.after").alias("after"),
                            col("value_json.source.ts_ms").alias("source_ts_ms"),
                            col("value_json.source.lsn").alias("source_lsn"),
                            col("value_json.source.txId").alias("source_txId"),
                            col("value_json.source.table").alias("source_table"),
                            col("value_json.source.snapshot").alias("source_snapshow"),
                            current_timestamp().alias("ingestion_ts")

)
flatened_event.printSchema()

root
 |-- kafka_topic: string (nullable = true)
 |-- kafka_partition: integer (nullable = true)
 |-- kafka_offset: long (nullable = true)
 |-- kafka_timestamp: timestamp (nullable = true)
 |-- operation: string (nullable = true)
 |-- before: string (nullable = true)
 |-- after: string (nullable = true)
 |-- source_ts_ms: long (nullable = true)
 |-- source_lsn: long (nullable = true)
 |-- source_txId: long (nullable = true)
 |-- source_table: string (nullable = true)
 |-- source_snapshow: string (nullable = true)
 |-- ingestion_ts: timestamp (nullable = false)



In [18]:
flatened_event2 = df_json.select(                                
                            col("op").alias("operation"),
                            col("before").alias("before"),
                            col("after").alias("after"),
                            col("source.ts_ms").alias("source_ts_ms"),
                            col("source.lsn").alias("source_lsn"),
                            col("source.txId").alias("source_txId"),
                            col("source.table").alias("source_table"),
                            col("source.snapshot").alias("source_snapshow"),
                            current_timestamp().alias("ingestion_ts")

)
flatened_event2.printSchema()

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `op` cannot be resolved. Did you mean one of the following? [`operation`, `kafka_topic`, `source_lsn`, `source_txid`, `ingestion_ts`].;
'Project ['op AS operation#325, 'before AS before#326, 'after AS after#327, 'source.ts_ms AS source_ts_ms#328, 'source.lsn AS source_lsn#329, 'source.txId AS source_txId#330, 'source.table AS source_table#331, 'source.snapshot AS source_snapshow#332, current_timestamp() AS ingestion_ts#333]
+- Relation [operation#96,source_ts_ms#97L,source_lsn#98L,source_txid#99L,source_table#100,source_snapshot#101,before_payload#102,after_payload#103,kafka_topic#104,kafka_partition#105,kafka_offset#106L,kafka_timestamp#107,ingestion_ts#108] parquet


In [ ]:
tables = flatened_event2.select(col("kafka_topic")).distinct()
tables = tables.toPandas()
tables

In [ ]:
df_payments_topic = flatened_event2.filter(col("source_table").contains("payments"))
df_reviews_topic = flatened_event2.filter(col("source_table").contains("reviews"))
df_reviews_topic = flatened_event2.filter(col("source_table").contains("orders"))
df_orders_topic = flatened_event2.filter(col("source_table").contains("orders"))
df_order_items_topic = flatened_event2.filter(col("source_table").contains("order_items"))
df_order_status_event_topic = flatened_event2.filter(col("source_table").contains("order_status_event"))

df_order_status_event_topic.show()

In [ ]:
TOPIC_TO_TABLES

In [ ]:

def write_to_bronze(df, batch_id):
    tables_list = [r["source_table"] for r in df.select("source_table").distinct().collect()]
    for t in tables_list:
        print(f"table: {t}, Batch id:", str(batch_id))
        print(f"kafka current offset:",kafka_offset)
        df_filterd = df.filter(col("source_table").contains(t))
        if df_reviews_topic.isEmpty():
            continue
        (
            df_filterd.write
                .format('iceberg')
                .mode("append")
                .toTable(f"lake.bronze.{t}")
                .save()
        )
        (
            df_filterd.write
                .format('console')
                .mode("append")
                .save()
        )
    

In [ ]:
(flatened_event.writeStream
                .foreachBatch(write_to_bronze)
                .outputMode("append")
                .trigger(processingTime = "10 seconds")
                .start()
                .awaitTermination()
                 

)            

In [ ]:
print(spark.version)
print(spark.sparkContext.appName)
print(spark.sparkContext._jvm.org.apache.spark.SparkContext.getOrCreate().isStopped())